In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Demucs Source Separation Quality Benchmark\n",
    "\n",
    "This notebook evaluates the quality of Demucs source separation using `mir_eval` metrics: SDR, SIR, SAR.  \n",
    "We will separate a mixture and compare the estimated stems against ground truth stems (if available).\n",
    "\n",
    "## Prerequisites\n",
    "- Install required packages: `pip install demucs mir_eval librosa`\n",
    "- Have a mixture audio file and its ground truth stems (e.g., from [MUSDB18](https://sigsep.github.io/datasets/musdb.html)).  \n",
    "  If you don't have ground truth, we'll create a synthetic example."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import numpy as np\n",
    "import librosa\n",
    "import mir_eval\n",
    "from pathlib import Path\n",
    "import soundfile as sf\n",
    "from demucs import pretrained\n",
    "from demucs.apply import apply_model\n",
    "from demucs.audio import AudioFile\n",
    "import torch"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Option 1: Use a synthetic mixture (no ground truth needed)\n",
    "We'll create a simple mixture by summing two synthetic stems (e.g., a sine wave and noise). This lets us compute exact metrics."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "def create_synthetic_stems(duration=5.0, sr=44100):\n",
    "    t = np.linspace(0, duration, int(sr*duration), endpoint=False)\n",
    "    # Stem 1: 440 Hz sine wave\n",
    "    stem1 = 0.5 * np.sin(2 * np.pi * 440 * t)\n",
    "    # Stem 2: white noise\n",
    "    stem2 = 0.3 * np.random.randn(len(t))\n",
    "    mixture = stem1 + stem2\n",
    "    return mixture, [stem1, stem2], ['sine', 'noise'], sr\n",
    "\n",
    "mix, stems, stem_names, sr = create_synthetic_stems()\n",
    "\n",
    "# Save stems temporarily\n",
    "sf.write('tmp_mix.wav', mix, sr)\n",
    "for i, stem in enumerate(stems):\n",
    "    sf.write(f'tmp_stem_{stem_names[i]}.wav', stem, sr)\n",
    "print(\"Synthetic stems created.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Option 2: Use a real mixture with ground truth (e.g., MUSDB18)\n",
    "If you have a MUSDB18 track, set the paths below."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Example paths (adjust to your data)\n",
    "mix_path = \"path/to/mixture.wav\"\n",
    "gt_stems = {\n",
    "    'drums': \"path/to/drums.wav\",\n",
    "    'bass': \"path/to/bass.wav\",\n",
    "    'other': \"path/to/other.wav\",\n",
    "    'vocals': \"path/to/vocals.wav\"\n",
    "}\n",
    "\n",
    "# Uncomment to use real data\n",
    "# mix, sr = librosa.load(mix_path, sr=44100, mono=False)\n",
    "# stems = [librosa.load(p, sr=44100, mono=False)[0] for p in gt_stems.values()]\n",
    "# stem_names = list(gt_stems.keys())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Run Demucs separation"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load Demucs model (use CPU for this demo)\n",
    "device = 'cpu'\n",
    "model = pretrained.get_model('htdemucs')\n",
    "model.to(device)\n",
    "model.eval()\n",
    "\n",
    "# Load mixture (Demucs expects shape (channels, samples))\n",
    "wav = AudioFile('tmp_mix.wav').read(\n",
    "    streams=0,\n",
    "    samplerate=model.samplerate,\n",
    "    channels=model.audio_channels\n",
    ")\n",
    "if wav.shape[1] != model.audio_channels:\n",
    "    wav = wav.T\n",
    "wav_tensor = torch.from_numpy(wav).float().to(device).unsqueeze(0)\n",
    "\n",
    "with torch.no_grad():\n",
    "    estimates = apply_model(model, wav_tensor, device=device)[0]\n",
    "estimates = estimates.cpu().numpy()  # shape (sources, channels, samples)\n",
    "\n",
    "# Demucs source order: ['drums', 'bass', 'other', 'vocals']\n",
    "estimated_stems = {name: estimates[i] for i, name in enumerate(model.sources)}\n",
    "print(\"Separation completed.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Align and evaluate\n",
    "We need to ensure the reference stems and estimated stems have the same length and channel count."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# For synthetic example, we have two stems: 'sine' and 'noise'. \n",
    "# Demucs outputs 4 stems; we need to map them appropriately.\n",
    "# Here we'll just compare the 'other' stem (index 2) to noise, and 'vocals' (index 3) to sine.\n",
    "ref_list = [stems[1], stems[0]]  # noise, sine\n",
    "est_list = [estimated_stems['other'], estimated_stems['vocals']]\n",
    "\n",
    "# Convert to mono if stereo\n",
    "ref_mono = [r if r.ndim==1 else r.mean(axis=0) for r in ref_list]\n",
    "est_mono = [e if e.ndim==1 else e.mean(axis=0) for e in est_list]\n",
    "\n",
    "# Trim to same length\n",
    "min_len = min(r.shape[-1] for r in ref_mono + est_mono)\n",
    "ref_mono = [r[..., :min_len] for r in ref_mono]\n",
    "est_mono = [e[..., :min_len] for e in est_mono]\n",
    "\n",
    "# Stack into 2D arrays (sources x samples)\n",
    "ref_stack = np.stack(ref_mono)\n",
    "est_stack = np.stack(est_mono)\n",
    "\n",
    "# Compute metrics\n",
    "sdr, sir, sar, _ = mir_eval.separation.bss_eval_sources(ref_stack, est_stack)\n",
    "\n",
    "print(\"\\n--- Evaluation Results ---\")\n",
    "for i, name in enumerate(['noise', 'sine']):\n",
    "    print(f\"{name} stem:\")\n",
    "    print(f\"  SDR: {sdr[i]:.2f} dB\")\n",
    "    print(f\"  SIR: {sir[i]:.2f} dB\")\n",
    "    print(f\"  SAR: {sar[i]:.2f} dB\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Interpretation\n",
    "- **SDR** (Signal to Distortion Ratio): overall quality (higher better).\n",
    "- **SIR** (Signal to Interference Ratio): how well other sources are suppressed.\n",
    "- **SAR** (Signal to Artifacts Ratio): absence of artifacts.\n",
    "\n",
    "For well‑separated signals, SDR > 10 dB is good, > 20 dB excellent."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Cleanup temporary files"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "os.remove('tmp_mix.wav')\n",
    "for name in stem_names:\n",
    "    os.remove(f'tmp_stem_{name}.wav')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.11.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}